In [1]:
# Run this first to train and save the model
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import joblib
import os
import shutil

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

# =========================
# CONFIGURATION
# =========================
# Set this to where you want to save the model (for copying to planner)
MODEL_EXPORT_FOLDER = "saved_model"  # All model files will go here
os.makedirs(MODEL_EXPORT_FOLDER, exist_ok=True)

# =========================
# 1. LOAD DATA
# =========================
print("Loading data...")
df = pd.read_csv("failure_risk_dataset.csv")

# Remove leakage
df = df.drop(columns=["passed_tests"])

if "test_error" in df.columns:
    df = df.drop(columns=["test_error"])

target = "failed"
df = df.dropna(subset=[target])

print(f"Data loaded: {len(df)} samples")

# =========================
# 2. FEATURES
# =========================
print("Engineering features...")
X = df.drop(columns=[target])
y = df[target]

# Store original test data for output
original_data = df.copy()

# Encode target
if y.dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)
    joblib.dump(le, f"{MODEL_EXPORT_FOLDER}/label_encoder.pkl")

# Feature engineering
X["complexity_per_len"] = X["avg_complexity"] / (X["code_len"] + 1)
X["ast_per_len"] = X["ast_nodes"] / (X["code_len"] + 1)
X["log_code_len"] = np.log1p(X["code_len"])

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Fill missing
X = X.fillna(X.median())

print(f"Feature count: {X.shape[1]}")

# =========================
# 3. SPLIT
# =========================
print("Splitting data...")
X_temp, X_test, y_temp, y_test, idx_temp, idx_test = train_test_split(
    X, y, range(len(X)), test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_temp, y_temp, idx_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# =========================
# 4. SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, f"{MODEL_EXPORT_FOLDER}/scaler.pkl")

# Save feature names for later inference
feature_names = X.columns.tolist()
joblib.dump(feature_names, f"{MODEL_EXPORT_FOLDER}/feature_names.pkl")

# =========================
# 5. TENSORS
# =========================
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(np.array(y_train), dtype=torch.float32).view(-1,1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(np.array(y_val), dtype=torch.float32).view(-1,1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(np.array(y_test), dtype=torch.float32).view(-1,1)

# =========================
# 6. DATALOADER
# =========================
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32,
    shuffle=True
)

# =========================
# 7. MODEL
# =========================
class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = ANN(X_train.shape[1])

# =========================
# 8. LOSS + OPTIMIZER
# =========================
pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# =========================
# 9. TRAINING
# =========================
print("\nTraining model...")
best_val_loss = float('inf')
patience = 7
counter = 0
best_model_state = None

for epoch in range(100):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Val Loss: {val_loss.item():.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_state = model.state_dict().copy()
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load the best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Loaded best model")

# =========================
# 10. EVALUATION
# =========================
print("\nEvaluating model...")
model.eval()
with torch.no_grad():
    y_test_logits = model(X_test_t)
    y_test_prob = torch.sigmoid(y_test_logits).numpy()

# =========================
# 11. THRESHOLD TUNING
# =========================
best_acc = 0
best_t = 0.5

for t in np.arange(0.3, 0.8, 0.05):
    preds = (y_test_prob > t).astype(int)
    acc = accuracy_score(y_test, preds)
    if acc > best_acc:
        best_acc = acc
        best_t = t

print(f"\nBest Threshold: {best_t:.3f}")
print(f"Best Accuracy: {best_acc:.4f}")

# =========================
# 12. FINAL METRICS
# =========================
y_pred = (y_test_prob > best_t).astype(int)
print("\n===== FINAL METRICS =====")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_test_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================
# 13. CREATE TEST PREDICTIONS CSV WITH RISK SCORES
# =========================
print("\nCreating test predictions CSV with risk scores...")

# Get original test data rows
test_original_data = original_data.iloc[idx_test].copy()

# Add predictions and risk scores
test_original_data['actual_failed'] = y_test
test_original_data['predicted_failed'] = y_pred
test_original_data['risk_score'] = y_test_prob.flatten()  # Probability (0-1)
test_original_data['risk_percentage'] = test_original_data['risk_score'] * 100

# Add risk level categories based on probability
def get_risk_level(prob):
    if prob < 0.3:
        return 'Low'
    elif prob < 0.6:
        return 'Medium'
    elif prob < 0.8:
        return 'High'
    else:
        return 'Very High'

test_original_data['risk_level'] = test_original_data['risk_score'].apply(get_risk_level)

# Sort by risk score (highest first) to see most risky cases
test_original_data = test_original_data.sort_values('risk_score', ascending=False)

# Save to CSV
test_predictions_csv = f"{MODEL_EXPORT_FOLDER}/test_predictions_with_risk.csv"
test_original_data.to_csv(test_predictions_csv, index=False)
print(f"✅ Test predictions saved to: {test_predictions_csv}")

# Also create a summary statistics CSV
risk_summary = pd.DataFrame({
    'metric': [
        'Total test samples',
        'Low risk (<30%)',
        'Medium risk (30-60%)', 
        'High risk (60-80%)',
        'Very high risk (≥80%)',
        'Mean risk score',
        'Median risk score',
        'Std risk score',
        'Min risk score',
        'Max risk score'
    ],
    'value': [
        len(test_original_data),
        len(test_original_data[test_original_data['risk_score'] < 0.3]),
        len(test_original_data[(test_original_data['risk_score'] >= 0.3) & (test_original_data['risk_score'] < 0.6)]),
        len(test_original_data[(test_original_data['risk_score'] >= 0.6) & (test_original_data['risk_score'] < 0.8)]),
        len(test_original_data[test_original_data['risk_score'] >= 0.8]),
        f"{test_original_data['risk_score'].mean():.4f}",
        f"{test_original_data['risk_score'].median():.4f}",
        f"{test_original_data['risk_score'].std():.4f}",
        f"{test_original_data['risk_score'].min():.4f}",
        f"{test_original_data['risk_score'].max():.4f}"
    ]
})

risk_summary_csv = f"{MODEL_EXPORT_FOLDER}/risk_score_summary.csv"
risk_summary.to_csv(risk_summary_csv, index=False)
print(f"✅ Risk summary saved to: {risk_summary_csv}")

# =========================
# 14. SAVE FINAL MODEL
# =========================
print("\nSaving model artifacts...")

# Save the final model
final_model_path = f"{MODEL_EXPORT_FOLDER}/failure_risk_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': X_train.shape[1],
    'best_threshold': best_t,
    'model_architecture': {
        'input_dim': X_train.shape[1],
        'hidden_layers': [256, 128, 64],
        'output_dim': 1
    },
    'metrics': {
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_test_prob),
        'best_threshold': best_t
    }
}, final_model_path)

# Save the best threshold
with open(f"{MODEL_EXPORT_FOLDER}/best_threshold.txt", "w") as f:
    f.write(str(best_t))

# Save model info (optional, for documentation)
model_info = {
    'input_features': feature_names,
    'input_dim': X_train.shape[1],
    'n_features': len(feature_names),
    'threshold': best_t,
    'accuracy': accuracy_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_test_prob),
    'train_samples': len(X_train),
    'val_samples': len(X_val),
    'test_samples': len(X_test),
    'class_distribution': {
        'failed': int(y.sum()),
        'success': int(len(y) - y.sum())
    }
}
joblib.dump(model_info, f"{MODEL_EXPORT_FOLDER}/model_info.pkl")

# =========================
# 15. CREATE ZIP FOR EASY COPY
# =========================
print("\n" + "="*50)
print("MODEL EXPORT COMPLETE")
print("="*50)

print(f"\n✅ All model artifacts saved to: '{MODEL_EXPORT_FOLDER}/'")

print("\n📁 Files in this folder:")
for file in os.listdir(MODEL_EXPORT_FOLDER):
    file_path = os.path.join(MODEL_EXPORT_FOLDER, file)
    size = os.path.getsize(file_path)
    print(f"   - {file} ({size/1024:.1f} KB)")

print("\n📋 To use in your planner:")
print(f"   1. Copy the entire '{MODEL_EXPORT_FOLDER}' folder")
print(f"   2. Paste it into your planner's working directory")
print(f"   3. Initialize the risk estimator with:")
print(f"      risk_estimator = ANNRiskEstimator()")
print(f"      # It will automatically load from '{MODEL_EXPORT_FOLDER}/'")

print("\n📊 Test predictions with risk scores available in:")
print(f"   - {test_predictions_csv}")
print(f"   - {risk_summary_csv}")

# Display first few rows of predictions
print("\n🔍 Preview of test predictions with risk scores:")
print(test_original_data[['risk_score', 'risk_level', 'actual_failed', 'predicted_failed']].head(10).to_string())

# Create a simple copy script
copy_instructions = f"""
# QUICK COPY COMMANDS:

# Option 1: Copy entire folder
cp -r {MODEL_EXPORT_FOLDER} /path/to/your/planner/

# Option 2: Copy individual files
cp {MODEL_EXPORT_FOLDER}/failure_risk_model.pth /path/to/your/planner/models/
cp {MODEL_EXPORT_FOLDER}/scaler.pkl /path/to/your/planner/
cp {MODEL_EXPORT_FOLDER}/feature_names.pkl /path/to/your/planner/
cp {MODEL_EXPORT_FOLDER}/best_threshold.txt /path/to/your/planner/models/
cp {MODEL_EXPORT_FOLDER}/test_predictions_with_risk.csv /path/to/your/planner/
cp {MODEL_EXPORT_FOLDER}/risk_score_summary.csv /path/to/your/planner/
"""

print("\n" + copy_instructions)

print("\n🎉 Done! Your model is ready to be copied to the planner.")

Loading data...
Data loaded: 1306 samples
Engineering features...
Feature count: 9
Splitting data...
Train: 783, Val: 261, Test: 262

Training model...
Epoch 10, Val Loss: 0.3660
Epoch 20, Val Loss: 0.3676
Early stopping at epoch 22
Loaded best model

Evaluating model...

Best Threshold: 0.300
Best Accuracy: 0.7786

===== FINAL METRICS =====
Accuracy: 0.7786
ROC-AUC: 0.8490

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.63      0.67        92
           1       0.81      0.86      0.83       170

    accuracy                           0.78       262
   macro avg       0.76      0.74      0.75       262
weighted avg       0.77      0.78      0.78       262


Creating test predictions CSV with risk scores...
✅ Test predictions saved to: saved_model/test_predictions_with_risk.csv
✅ Risk summary saved to: saved_model/risk_score_summary.csv

Saving model artifacts...

MODEL EXPORT COMPLETE

✅ All model artifacts saved to: 'saved